In [0]:
import os, sys
import logging
import datetime
from datetime import datetime
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from typing import Optional
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sum,
    col,
    sha2,
    substring,
    to_date,
    when,
    broadcast,
    trim,
    row_number,
    desc
)

def load_landed_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> SparkDataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def load_landed_portfolio_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> SparkDataFrame:
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

##### 1. Create **business_plan_rawdata** table

In [0]:
def businessPlanCorpBankruptcy(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smevbusinessplandatacorpbankruptsy ====================================

    business_plan_corp_bankruptcy_extra_filters = None
    if business_plan_corp_bankruptcy_extra_filters is not None:
        business_plan_corp_bankruptcy_extra_filters.append(date_filter)
        business_plan_corp_bankruptcy_final_filters = " AND ".join(
            business_plan_corp_bankruptcy_extra_filters
        )
    else:
        business_plan_corp_bankruptcy_final_filters = date_filter

    business_plan_corp_bankruptcy_selected_filters = [
        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "CollectionPlan_Month",
        "CollectionPlan_Amount",
        "PortfolioID",
        "BusinessDate",
        ]

    # Keys of the table: **BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, CollectionPlan_Month**
    business_plan_corp_bankruptcy = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandatacorpbankruptsy",
        business_plan_corp_bankruptcy_final_filters,
        business_plan_corp_bankruptcy_selected_filters,
        )

    agg_business_plan_corp_bankruptcy = (
        business_plan_corp_bankruptcy.groupBy(
            "BusinessPlan_ID",
            "AgreementNumber",
            "BusinessPlan_Status",
            "BusinessPlanStrategy_StrategyType",
            "BusinessPlanStrategy_Code",
            "CollectionPlan_Month"
        )
        .agg(sum("CollectionPlan_Amount").alias("CollectionPlan_Total_Amount"))
    )

    window_spec = Window.partitionBy(
        "BusinessPlan_ID",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Code",
        "CollectionPlan_Month"
        ).orderBy("CollectionPlan_Amount")

    dedup_business_plan_corp_bankruptcy = (
        business_plan_corp_bankruptcy.withColumn("rank", row_number().over(window_spec)).filter(col("rank") == 1)
        )

    business_plan_corp_bankruptcy_join_condition = (
        (col("dedup.BusinessPlan_ID") == col("agg.BusinessPlan_ID"))
            &(col("dedup.AgreementNumber") == col("agg.AgreementNumber"))
            &(col("dedup.BusinessPlan_Status") == col("agg.BusinessPlan_Status"))
            &(col("dedup.BusinessPlanStrategy_StrategyType") == col("agg.BusinessPlanStrategy_StrategyType"))
            &(col("dedup.BusinessPlanStrategy_Code") == col("agg.BusinessPlanStrategy_Code"))
            &(col("dedup.CollectionPlan_Month") == col("agg.CollectionPlan_Month"))
    )

    business_plan_corp_bankruptcy_df = (
        dedup_business_plan_corp_bankruptcy.alias('dedup').join(
            agg_business_plan_corp_bankruptcy.alias("agg"),
            business_plan_corp_bankruptcy_join_condition,
            how="left"
        ).select(
            "dedup.*",
            "agg.CollectionPlan_Total_Amount"
            )
    )

    # ================== business_plan_corp_bankruptcy table schema ==================

    business_plan_corp_bankruptcy = business_plan_corp_bankruptcy_df.select(
        business_plan_corp_bankruptcy_df.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        business_plan_corp_bankruptcy_df.DebtorCDI.cast("String"),
        business_plan_corp_bankruptcy_df.AgreementNumber.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlan_Status.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        business_plan_corp_bankruptcy_df.BusinessPlan_SolutionType.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlan_Comments.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Status.cast("String"),
        business_plan_corp_bankruptcy_df.Confidence.cast("String"),
        business_plan_corp_bankruptcy_df.ProposedBy.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlanStrategy_StrategyType.cast("String"),
        business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Comments.cast("String"),
        business_plan_corp_bankruptcy_df.LastModifiedDate.cast("Date"),
        business_plan_corp_bankruptcy_df.CollectionPlan_Month.cast("Date"),
        business_plan_corp_bankruptcy_df.CollectionPlan_Total_Amount.alias("CollectionPlan_Amount").cast("Decimal(19, 2)"), # Important
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        business_plan_corp_bankruptcy_df.PortfolioID.cast("Int"),
        business_plan_corp_bankruptcy_df.BusinessDate.cast("Date"),
        lit("Business_Plan_Corp_Bankruptcy").cast("String").alias("tbl"),
        )

    return business_plan_corp_bankruptcy

In [0]:
def businessPlanToolForc(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smevbusinessplandataforccbsnonre ====================================

    nonre_forc_extra_filters = None
    if nonre_forc_extra_filters is not None:
        nonre_forc_extra_filters.append(date_filter)
        nonre_forc_final_filters = " AND ".join(
            nonre_forc_extra_filters
        )
    else:
        nonre_forc_final_filters = date_filter

    nonre_forc_selected_filters = [
        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "TimeToRecovery",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    nonre_forc = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandataforccbsnonre",
        nonre_forc_final_filters,
        nonre_forc_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandataforccbsre3rdparty =======================================

    re_3rdparty_forc_extra_filters = None
    if re_3rdparty_forc_extra_filters is not None:
        re_3rdparty_forc_extra_filters.append(date_filter)
        re_3rdparty_forc_final_filters = " AND ".join(
            re_3rdparty_forc_extra_filters
        )
    else:
        re_3rdparty_forc_final_filters = date_filter

    re_3rdparty_forc_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "AuctionPrice",
        "TimeToAuction",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    re_3rdparty_forc = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandataforccbsre3rdparty",
        re_3rdparty_forc_final_filters,
        re_3rdparty_forc_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandataforccbsreohold =======================================

    reo_hold_forc_extra_filters = None
    if reo_hold_forc_extra_filters is not None:
        reo_hold_forc_extra_filters.append(date_filter)
        reo_hold_forc_final_filters = " AND ".join(
            reo_hold_forc_extra_filters
        )
    else:
        reo_hold_forc_final_filters = date_filter

    reo_hold_forc_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "AuctionPrice",
        "TimeToAuction",
        "TimeToSell",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    reo_hold_forc = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandataforccbsreohold",
        reo_hold_forc_final_filters,
        reo_hold_forc_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandataforccbsreosale =======================================

    reo_sale_forc_extra_filters = None
    if reo_sale_forc_extra_filters is not None:
        reo_sale_forc_extra_filters.append(date_filter)
        reo_sale_forc_final_filters = " AND ".join(
            reo_sale_forc_extra_filters
        )
    else:
        reo_sale_forc_final_filters = date_filter

    reo_sale_forc_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "AuctionPrice",
        "TimeToAuction",
        "TimeToSell",
        #"TimeToAuctionRM",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    reo_sale_forc = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandataforccbsreosale",
        reo_sale_forc_final_filters,
        reo_sale_forc_selected_filters,
        )

    # ================== nonre_forc table schema ==================

    target_nonre_forc = nonre_forc.select(
        nonre_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        nonre_forc.DebtorCDI.cast("String"),
        nonre_forc.AgreementNumber.cast("String"),
        nonre_forc.BusinessPlan_Status.cast("String"),
        nonre_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        nonre_forc.BusinessPlan_SolutionType.cast("String"),
        nonre_forc.BusinessPlan_Comments.cast("String"),
        nonre_forc.BusinessPlanStrategy_Status.cast("String"),
        nonre_forc.Confidence.cast("String"),
        nonre_forc.ProposedBy.cast("String"),
        nonre_forc.BusinessPlanStrategy_StrategyType.cast("String"),
        nonre_forc.BusinessPlanStrategy_Comments.cast("String"),
        nonre_forc.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        nonre_forc.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
        nonre_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        nonre_forc.PortfolioID.cast("Int"),
        nonre_forc.BusinessDate.cast("Date"),
        lit("Business_Plan_Forc_CBS_NonRE").cast("String").alias("tbl"),
    )

    # ================== re_3rdparty_forc table schema ==================

    target_re_3rdparty_forc = re_3rdparty_forc.select(
        re_3rdparty_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
        re_3rdparty_forc.DebtorCDI.cast("String"),
        re_3rdparty_forc.AgreementNumber.cast("String"),
        re_3rdparty_forc.BusinessPlan_Status.cast("String"),
        re_3rdparty_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        re_3rdparty_forc.BusinessPlan_SolutionType.cast("String"),
        re_3rdparty_forc.BusinessPlan_Comments.cast("String"),
        re_3rdparty_forc.BusinessPlanStrategy_Status.cast("String"),
        re_3rdparty_forc.Confidence.cast("String"),
        re_3rdparty_forc.ProposedBy.cast("String"),
        re_3rdparty_forc.BusinessPlanStrategy_StrategyType.cast("String"),
        re_3rdparty_forc.BusinessPlanStrategy_Comments.cast("String"),
        re_3rdparty_forc.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        re_3rdparty_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
        re_3rdparty_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        re_3rdparty_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        re_3rdparty_forc.PortfolioID.cast("Int"),
        re_3rdparty_forc.BusinessDate.cast("Date"),
        lit("Business_Plan_Forc_CBS_RE_3rdParty").cast("String").alias("tbl"),
        )

    # ================== reo_hold_forc table schema ==================

    target_reo_hold_forc = reo_hold_forc.select(
        reo_hold_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
        reo_hold_forc.DebtorCDI.cast("String"),
        reo_hold_forc.AgreementNumber.cast("String"),
        reo_hold_forc.BusinessPlan_Status.cast("String"),
        reo_hold_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        reo_hold_forc.BusinessPlan_SolutionType.cast("String"),
        reo_hold_forc.BusinessPlan_Comments.cast("String"),
        reo_hold_forc.BusinessPlanStrategy_Status.cast("String"),
        reo_hold_forc.Confidence.cast("String"),
        reo_hold_forc.ProposedBy.cast("String"),
        reo_hold_forc.BusinessPlanStrategy_StrategyType.cast("String"),
        reo_hold_forc.BusinessPlanStrategy_Comments.cast("String"),
        reo_hold_forc.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        reo_hold_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
        reo_hold_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
        reo_hold_forc.TimeToSell.cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        reo_hold_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        reo_hold_forc.PortfolioID.cast("Int"),
        reo_hold_forc.BusinessDate.cast("Date"),
        lit("Business_Plan_Forc_CBS_REO_Hold").cast("String").alias("tbl"),
    )

    # ================== reo_sale_forc table schema ==================

    target_reo_sale_forc = reo_sale_forc.select(
        reo_sale_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
        reo_sale_forc.DebtorCDI.cast("String"),
        reo_sale_forc.AgreementNumber.cast("String"),
        reo_sale_forc.BusinessPlan_Status.cast("String"),
        reo_sale_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        reo_sale_forc.BusinessPlan_SolutionType.cast("String"),
        reo_sale_forc.BusinessPlan_Comments.cast("String"),
        reo_sale_forc.BusinessPlanStrategy_Status.cast("String"),
        reo_sale_forc.Confidence.cast("String"),
        reo_sale_forc.ProposedBy.cast("String"),
        reo_sale_forc.BusinessPlanStrategy_StrategyType.cast("String"),
        reo_sale_forc.BusinessPlanStrategy_Comments.cast("String"),
        reo_sale_forc.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        reo_sale_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
        reo_sale_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
        reo_sale_forc.TimeToSell.cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        reo_sale_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        reo_sale_forc.PortfolioID.cast("Int"),
        reo_sale_forc.BusinessDate.cast("Date"),
        lit("Business_Plan_Forc_CBS_REO_Sale").cast("String").alias("tbl"),
    )

    tool_forc_cbs = (
        target_nonre_forc
        .union(target_re_3rdparty_forc)
        .union(target_reo_hold_forc)
        .union(target_reo_sale_forc)
        )
    return tool_forc_cbs

In [0]:
def businessPlanToolVol(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smevbusinessplandatavolcbsnonre ====================================

    nonre_vol_extra_filters = None
    if nonre_vol_extra_filters is not None:
        nonre_vol_extra_filters.append(date_filter)
        nonre_vol_final_filters = " AND ".join(
            nonre_vol_extra_filters
        )
    else:
        nonre_vol_final_filters = date_filter

    nonre_vol_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "TimeToRecovery",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    nonre_vol = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandatavolcbsnonre",
        nonre_vol_final_filters,
        nonre_vol_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandatavolcbsre3rdparty =======================================

    re_3rdparty_vol_extra_filters = None
    if re_3rdparty_vol_extra_filters is not None:
        re_3rdparty_vol_extra_filters.append(date_filter)
        re_3rdparty_vol_final_filters = " AND ".join(
            re_3rdparty_vol_extra_filters
        )
    else:
        re_3rdparty_vol_final_filters = date_filter

    re_3rdparty_vol_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "TimeToSell",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    re_3rdparty_vol = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandatavolcbsre3rdparty",
        re_3rdparty_vol_final_filters,
        re_3rdparty_vol_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandatavolcbsreohold =======================================

    reo_hold_vol_extra_filters = None
    if reo_hold_vol_extra_filters is not None:
        reo_hold_vol_extra_filters.append(date_filter)
        reo_hold_vol_final_filters = " AND ".join(
            reo_hold_vol_extra_filters
        )
    else:
        reo_hold_vol_final_filters = date_filter

    reo_hold_vol_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "TimeToSell",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    reo_hold_vol = load_landed_delta_table(
        spark,
        catalog_landed, 
        "cepalplatform_smevbusinessplandatavolcbsreohold",
        reo_hold_vol_final_filters,
        reo_hold_vol_selected_filters,
        )

    # ==================================== cepalplatform_smevbusinessplandatavolcbsreosale =======================================

    reo_sale_vol_extra_filters = None
    if reo_sale_vol_extra_filters is not None:
        reo_sale_vol_extra_filters.append(date_filter)
        reo_sale_vol_final_filters = " AND ".join(
            reo_sale_vol_extra_filters
        )
    else:
        reo_sale_vol_final_filters = date_filter

    reo_sale_vol_selected_filters = [

        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "TimeToSell",
        "NetRecoveryAmount",
        "BusinessDate"
    ]

    reo_sale_vol = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandatavolcbsreosale",
        reo_sale_vol_final_filters,
        reo_sale_vol_selected_filters,
        )

    # ================== nonre_vol table schema ==================

    target_nonre_vol = nonre_vol.select(
        nonre_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        nonre_vol.DebtorCDI.cast("String"),
        nonre_vol.AgreementNumber.cast("String"),
        nonre_vol.BusinessPlan_Status.cast("String"),
        nonre_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        nonre_vol.BusinessPlan_SolutionType.cast("String"),
        nonre_vol.BusinessPlan_Comments.cast("String"),
        nonre_vol.BusinessPlanStrategy_Status.cast("String"),
        nonre_vol.Confidence.cast("String"),
        nonre_vol.ProposedBy.cast("String"),
        nonre_vol.BusinessPlanStrategy_StrategyType.cast("String"),
        nonre_vol.BusinessPlanStrategy_Comments.cast("String"),
        nonre_vol.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        nonre_vol.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
        nonre_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        nonre_vol.PortfolioID.cast("Int"),
        nonre_vol.BusinessDate.cast("Date"),
        lit("Business_Plan_Vol_CBS_RE_3rdParty").cast("String").alias("tbl"),
    )

    # ================== re_3rdparty_vol table schema ==================

    target_re_3rdparty_vol = re_3rdparty_vol.select(
        re_3rdparty_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        re_3rdparty_vol.DebtorCDI.cast("String"),
        re_3rdparty_vol.AgreementNumber.cast("String"),
        re_3rdparty_vol.BusinessPlan_Status.cast("String"),
        re_3rdparty_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        re_3rdparty_vol.BusinessPlan_SolutionType.cast("String"),
        re_3rdparty_vol.BusinessPlan_Comments.cast("String"),
        re_3rdparty_vol.BusinessPlanStrategy_Status.cast("String"),
        re_3rdparty_vol.Confidence.cast("String"),
        re_3rdparty_vol.ProposedBy.cast("String"),
        re_3rdparty_vol.BusinessPlanStrategy_StrategyType.cast("String"),
        re_3rdparty_vol.BusinessPlanStrategy_Comments.cast("String"),
        re_3rdparty_vol.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        re_3rdparty_vol.TimeToSell.cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        re_3rdparty_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        re_3rdparty_vol.PortfolioID.cast("Int"),
        re_3rdparty_vol.BusinessDate.cast("Date"),
        lit("Business_Plan_Vol_CBS_RE_3rdParty").cast("String").alias("tbl"),
    )

    # ================== reo_hold_vol table schema ==================

    target_reo_hold_vol = reo_hold_vol.select(
        reo_hold_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        reo_hold_vol.DebtorCDI.cast("String"),
        reo_hold_vol.AgreementNumber.cast("String"),
        reo_hold_vol.BusinessPlan_Status.cast("String"),
        reo_hold_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        reo_hold_vol.BusinessPlan_SolutionType.cast("String"),
        reo_hold_vol.BusinessPlan_Comments.cast("String"),
        reo_hold_vol.BusinessPlanStrategy_Status.cast("String"),
        reo_hold_vol.Confidence.cast("String"),
        reo_hold_vol.ProposedBy.cast("String"),
        reo_hold_vol.BusinessPlanStrategy_StrategyType.cast("String"),
        reo_hold_vol.BusinessPlanStrategy_Comments.cast("String"),
        reo_hold_vol.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        reo_hold_vol.TimeToSell.cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        reo_hold_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        reo_hold_vol.PortfolioID.cast("Int"),
        reo_hold_vol.BusinessDate.cast("Date"),
        lit("Business_Plan_Vol_CBS_REO_Hold").cast("String").alias("tbl"),
    )

    # ================== reo_sale_vol table schema ===================================

    target_reo_sale_vol = reo_sale_vol.select(
        reo_sale_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        reo_sale_vol.DebtorCDI.cast("String"),
        reo_sale_vol.AgreementNumber.cast("String"),
        reo_sale_vol.BusinessPlan_Status.cast("String"),
        reo_sale_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        reo_sale_vol.BusinessPlan_SolutionType.cast("String"),
        reo_sale_vol.BusinessPlan_Comments.cast("String"),
        reo_sale_vol.BusinessPlanStrategy_Status.cast("String"),
        reo_sale_vol.Confidence.cast("String"),
        reo_sale_vol.ProposedBy.cast("String"),
        reo_sale_vol.BusinessPlanStrategy_StrategyType.cast("String"),
        reo_sale_vol.BusinessPlanStrategy_Comments.cast("String"),
        reo_sale_vol.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        reo_sale_vol.TimeToSell.cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        reo_sale_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        reo_sale_vol.PortfolioID.cast("Int"),
        reo_sale_vol.BusinessDate.cast("Date"),
        lit("Business_Plan_Vol_CBS_REO_Sale").cast("String").alias("tbl"),
    )
    tool_vol_cbs = (
        target_nonre_vol
        .union(target_re_3rdparty_vol)
        .union(target_reo_hold_vol)
        .union(target_reo_sale_vol)
    )

    return tool_vol_cbs


In [0]:
def businessPlanIndivBankruptcyStandStill(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smevbusinessplandataindivbankruptcy ====================================

    indiv_bankruptcy_extra_filters = None
    if indiv_bankruptcy_extra_filters is not None:
        indiv_bankruptcy_extra_filters.append(date_filter)
        indiv_bankruptcy_final_filters = " AND ".join(
            indiv_bankruptcy_extra_filters
        )
    else:
        indiv_bankruptcy_final_filters = date_filter

    indiv_bankruptcy_selected_filters = [
        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_CreationDate",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "NumberOfInstallments",
        "Installment",
        "PrepaymentMonth",
        "PrepaymentAmount",
        "PeriodicityID",
        "PortfolioID",
        "BusinessDate",
        ]

    indiv_bankruptcy = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandataindivbankruptcy",
        indiv_bankruptcy_final_filters,
        indiv_bankruptcy_selected_filters,
        )

    indiv_bankruptcy = indiv_bankruptcy.withColumn(
        "PeriodicityName",
        when(col("PeriodicityID") == 1, "Μηνιαία")
        .when(col("PeriodicityID") == 2, "Διμηνιαία")
        .when(col("PeriodicityID") == 3, "Τρίμηνη")
        .when(col("PeriodicityID") == 4, "Εξάμηνη")
        .when(col("PeriodicityID") == 5, "Ετήσια")
        .otherwise(None)
    )

    # ============================== cepalplatform_smevbusinessplandatastandstill ====================================

    stand_still_extra_filters = None
    if stand_still_extra_filters is not None:
        stand_still_extra_filters.append(date_filter)
        stand_still_final_filters = " AND ".join(
            stand_still_extra_filters
        )
    else:
        stand_still_final_filters = date_filter

    stand_still_selected_filters = [
        "BusinessPlan_ID",
        "DebtorCDI",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_Code",
        "BusinessPlan_CreationDate",
        "BusinessPlan_SolutionType",
        "BusinessPlan_Comments",
        "BusinessPlanStrategy_Status",
        "Confidence",
        "ProposedBy",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Comments",
        "LastModifiedDate",
        "PortfolioID",
        "BusinessDate",
        ]

    stand_still = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smevbusinessplandatastandstill",
        stand_still_final_filters,
        stand_still_selected_filters,
        )

    # ================== indiv_bankruptcy table schema ===================================

    target_indiv_bankruptcy = indiv_bankruptcy.select(
        indiv_bankruptcy.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        indiv_bankruptcy.DebtorCDI.cast("String"),
        indiv_bankruptcy.AgreementNumber.cast("String"),
        indiv_bankruptcy.BusinessPlan_Status.cast("String"),
        indiv_bankruptcy.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        indiv_bankruptcy.BusinessPlan_SolutionType.cast("String"),
        indiv_bankruptcy.BusinessPlan_Comments.cast("String"),
        indiv_bankruptcy.BusinessPlanStrategy_Status.cast("String"),
        indiv_bankruptcy.Confidence.cast("String"),
        indiv_bankruptcy.ProposedBy.cast("String"),
        indiv_bankruptcy.BusinessPlanStrategy_StrategyType.cast("String"),
        indiv_bankruptcy.BusinessPlanStrategy_Comments.cast("String"),
        indiv_bankruptcy.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        indiv_bankruptcy.NumberOfInstallments.cast("Int").alias("NumberOfInstallments"),
        indiv_bankruptcy.PrepaymentMonth.cast("Date").alias("PrepaymentMonth"),
        indiv_bankruptcy.PrepaymentAmount.cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        indiv_bankruptcy.PeriodicityName.cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        indiv_bankruptcy.PortfolioID.cast("Int"),
        indiv_bankruptcy.BusinessDate.cast("Date"),
        lit("Business_Plan_Indv_Bankcruptcy").cast("String").alias("tbl"),
    )

    # ================== stand_still table schema ===================================

    target_stand_still = stand_still.select(
        stand_still.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
        stand_still.DebtorCDI.cast("String"),
        stand_still.AgreementNumber.cast("String"),
        stand_still.BusinessPlan_Status.cast("String"),
        stand_still.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
        stand_still.BusinessPlan_SolutionType.cast("String"),
        stand_still.BusinessPlan_Comments.cast("String"),
        stand_still.BusinessPlanStrategy_Status.cast("String"),
        stand_still.Confidence.cast("String"),
        stand_still.ProposedBy.cast("String"),
        stand_still.BusinessPlanStrategy_StrategyType.cast("String"),
        stand_still.BusinessPlanStrategy_Comments.cast("String"),
        stand_still.LastModifiedDate.cast("Date"),
        lit(None).cast("Date").alias("CollectionPlan_Month"),
        lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
        lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
        lit(None).cast("Int").alias("NumberOfInstallments"),
        lit(None).cast("Date").alias("PrepaymentMonth"),
        lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
        lit(None).cast("Decimal(10, 4)").alias("Interest"),
        lit(None).cast("String").alias("PeriodicityID"),
        lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
        lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
        lit(None).cast("Int").alias("TimeToAuction"),
        lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
        lit(None).cast("Int").alias("TimeToSell"),
        lit(None).cast("Int").alias("TimeToRecovery"),
        lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
        lit(None).cast("String").alias("ImplementationStatus"),
        lit(None).cast("Date").alias("FirstInstallmentDate"),
        lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
        stand_still.PortfolioID.cast("Int"),
        stand_still.BusinessDate.cast("Date"),
        lit("Business_Plan_Stand_Still").cast("String").alias("tbl"),
    )

    indiv_bankruptcy_stand_still = target_indiv_bankruptcy.union(target_stand_still)
    return indiv_bankruptcy_stand_still

##### 2. Create **business_plan_lbspaymentschedule** table

In [0]:
def lbsPaymentsSchedule(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smebpsmlbspaymentschedule ====================================

    lbs_payments_schedule_extra_filters = None
    if lbs_payments_schedule_extra_filters is not None:
        lbs_payments_schedule_extra_filters.append(date_filter)
        lbs_payments_schedule_final_filters = " AND ".join(
            lbs_payments_schedule_extra_filters
        )
    else:
        lbs_payments_schedule_final_filters = date_filter

    lbs_payments_schedule_selected_filters = [
        "LBSParentID",
        "InstallmentSN",
        "InstallmentDate",
        "InstallmentTotalAmount",
        "InstallmentCapitalAmount",
        "InstallmentChargeAmount",
        "InstallmentN128Amount",
        "InstallmentTotalChargeAmount",
        "InstallmentRemainingCapitalAmount",
        "InstallmentNPVAmount",
        "InstallmentNPVCalculationDate",
        "CreationDate",
        "CreationUser",
        "IsActive",
        "BusinessDate",
        ]

    lbs_payments_schedule = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebpsmlbspaymentschedule",
        lbs_payments_schedule_final_filters,
        lbs_payments_schedule_selected_filters,
        )

    # Keys of the table: **LBSParentID, InstallmentSN, InstallmentDate (or CreationDate)**
    lbs_payments_schedule = lbs_payments_schedule.filter(col("IsActive") == True)

    # =============================== Target table =================================

    target_lbs_payments_schedule = lbs_payments_schedule.select(
        lbs_payments_schedule.LBSParentID.cast("Int"),
        lbs_payments_schedule.InstallmentSN.cast("Int"),
        lbs_payments_schedule.InstallmentDate.cast("Timestamp"),
        lbs_payments_schedule.InstallmentTotalAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentCapitalAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentChargeAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentN128Amount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentTotalChargeAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentRemainingCapitalAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentNPVAmount.cast("Decimal(18, 4)"),
        lbs_payments_schedule.InstallmentNPVCalculationDate.cast("Timestamp"),
        lbs_payments_schedule.CreationDate.cast("Timestamp"),
        lbs_payments_schedule.CreationUser.cast("String"),
        lbs_payments_schedule.IsActive.cast("Boolean"),
        lbs_payments_schedule.BusinessDate.cast("Date"),
    )
    return target_lbs_payments_schedule

##### 3. Create **business_plan_contract** table

In [0]:
def businessPlanContract(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smebusinessplancontract ====================================

    business_plan_contract_extra_filters = None
    if business_plan_contract_extra_filters is not None:
        business_plan_contract_extra_filters.append(date_filter)
        business_plan_contract_final_filters = " AND ".join(
            business_plan_contract_extra_filters
        )
    else:
        business_plan_contract_final_filters = date_filter

    business_plan_contract_selected_filters = [
        "ID",
        "AgreementNumber",
        "BusinessPlanStrategyManagementID",
        "creationdate",
        "CurrencyCode",
        "BusinessDate",
        ]

    business_plan_contract = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebusinessplancontract",
        business_plan_contract_final_filters,
        business_plan_contract_selected_filters,
        )

    # ============================== cepalplatform_smebusinessplancontractloan ====================================

    business_plan_contract_loan_extra_filters = None
    if business_plan_contract_loan_extra_filters is not None:
        business_plan_contract_loan_extra_filters.append(date_filter)
        business_plan_contract_loan_final_filters = " AND ".join(
            business_plan_contract_loan_extra_filters
        )
    else:
        business_plan_contract_loan_final_filters = date_filter

    business_plan_contract_loan_selected_filters = [
        "AccountNumber",
        "LoanStatus",
        "BusinessPlanContractID"
        ]

    business_plan_contract_loan = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebusinessplancontractloan",
        business_plan_contract_loan_final_filters,
        business_plan_contract_loan_selected_filters,
        )

    # ============================== cepalplatform_smebusinessplancontractcollateral ====================================

    business_plan_contract_collateral_extra_filters = None
    if business_plan_contract_collateral_extra_filters is not None:
        business_plan_contract_collateral_extra_filters.append(date_filter)
        business_plan_contract_collateral_final_filters = " AND ".join(
            business_plan_contract_collateral_extra_filters
        )
    else:
        business_plan_contract_collateral_final_filters = date_filter

    business_plan_contract_collateral_selected_filters = [
        "ID",
        "BusinessPlanContractID",
        "ABCollateralID",
        "ThesisCollateralID",
        "OwnershipID",
        "ABPropertyKey",
        "ThesisPropertyKey",
        ]

    business_plan_contract_collateral = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebusinessplancontractcollateral",
        business_plan_contract_collateral_final_filters,
        business_plan_contract_collateral_selected_filters,
        )

    # ============================== joins ====================================

    business_plan_contract_loan_collateral = (
        business_plan_contract
        .join(
            business_plan_contract_loan,
            business_plan_contract.ID == business_plan_contract_loan.BusinessPlanContractID,
            "left"
        )
        .join(
            business_plan_contract_collateral,
            business_plan_contract.ID == business_plan_contract_collateral.BusinessPlanContractID,
            "left"
        )                       
    )

    # ============================== business_contract schema ====================================

    target_business_plan_contract_loan_collateral = business_plan_contract_loan_collateral.select(
        business_plan_contract.ID.cast("Int"),
        business_plan_contract.BusinessPlanStrategyManagementID.cast("Int").alias("BusinessPlanStrategy_ID"),
        business_plan_contract.AgreementNumber.cast("String").alias("Agreement_Number"),
        business_plan_contract.creationdate.cast("Date").alias("CreationDate"),
        business_plan_contract_loan.AccountNumber.alias("Account_Number"),
        business_plan_contract_collateral.ABCollateralID.cast("String"),
        business_plan_contract_collateral.ThesisCollateralID.cast("String"),
        business_plan_contract_collateral.OwnershipID.cast("String"),
        business_plan_contract_collateral.ABPropertyKey.cast("String"),
        business_plan_contract_collateral.ThesisPropertyKey.cast("String"),
        business_plan_contract.CurrencyCode.cast("String"),
        business_plan_contract_loan.LoanStatus.cast("String"),
        business_plan_contract.BusinessDate.cast("Date"),
    )

    return target_business_plan_contract_loan_collateral

##### 4. Create **sme_business_plan** table

In [0]:
def businessPlan(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smebusinessplan ====================================

    business_plan_extra_filters = None
    if business_plan_extra_filters is not None:
        business_plan_extra_filters.append(date_filter)
        business_plan_final_filters = " AND ".join(
            business_plan_extra_filters
        )
    else:
        business_plan_final_filters = date_filter

    business_plan_selected_filters = [
        "ID",
        "StatusID",
        "SolutionTypeID",
        "DebtorCDI",
        "DebtorAFM",
        "DebtorBankGroupID",
        "IntentionDate",
        "CompletionTargetDate",
        "ApplicationDate",
        "Comments",
        "CreationDate",
        "CreationUser",
        "LastModifiedDate",
        "LastModifiedUser",
        "PortfolioID",
        "BusinessDate",
        ]

    business_plan = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebusinessplan",
        business_plan_final_filters,
        business_plan_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpstatus ====================================

    business_plan_status_extra_filters = None
    if business_plan_status_extra_filters is not None:
        business_plan_status_extra_filters.append(date_filter)
        business_plan_status_final_filters = " AND ".join(
            business_plan_status_extra_filters
        )
    else:
        business_plan_status_final_filters = date_filter

    business_plan_status_selected_filters = ["ID", "Label"]
    business_plan_status = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpstatus",
        business_plan_status_final_filters,
        business_plan_status_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpsolutiontype ====================================

    business_plan_solution_type_extra_filters = None
    if business_plan_solution_type_extra_filters is not None:
        business_plan_solution_type_extra_filters.append(date_filter)
        business_plan_solution_type_final_filters = " AND ".join(
            business_plan_solution_type_extra_filters
        )
    else:
        business_plan_solution_type_final_filters = date_filter

    business_plan_solution_type_selected_filters = ["ID", "Label"]
    business_plan_solution_type = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpsolutiontype",
        business_plan_solution_type_final_filters,
        business_plan_solution_type_selected_filters,
        )

    # ============================== joins =======================================

    business_plan_status_solution_type = (
        business_plan
        .join(
            business_plan_status,
            business_plan.StatusID == business_plan_status.ID,
            "left"
        )
        .join(
            business_plan_solution_type,
            business_plan.SolutionTypeID == business_plan_solution_type.ID,
            "left"
        )
    )

    # ============================== target table =======================================

    target_business_plan_status_solution_type = business_plan_status_solution_type.select(
        business_plan.ID.cast("Int"),
        business_plan.DebtorCDI.cast("String"),
        business_plan.DebtorAFM.cast("String"),
        business_plan.DebtorBankGroupID.cast("String").alias("GroupCode"),
        business_plan_status.Label.cast("String").alias("Business_plan_status"),
        business_plan_solution_type.Label.cast("String").alias("SolutionType"), 
        business_plan.IntentionDate.cast("Date"), 
        business_plan.CompletionTargetDate.cast("Date"),
        business_plan.ApplicationDate.cast("Date"),
        business_plan.Comments.cast("String"),
        business_plan.CreationDate.cast("Timestamp"),
        business_plan.CreationUser.cast("String"),
        business_plan.LastModifiedDate.cast("Timestamp"),
        business_plan.LastModifiedUser.cast("String"),
        business_plan.PortfolioID.cast("Int"),
        business_plan.BusinessDate.cast("Date"),
    )

    return target_business_plan_status_solution_type

##### 5. Create table: **sme_business_plan_strategy**

In [0]:
def businessPlanStrategy(spark: SparkSession, catalog_landed: str, run_date: str) -> SparkDataFrame:

    cluster_columns = ["BusinessDate"]
    date_filter = f"businessdate = {run_date!r}"

    # ============================== cepalplatform_smebusinessplanstrategymanagement ====================================

    business_plan_strategy_management_extra_filters = None
    if business_plan_strategy_management_extra_filters is not None:
        business_plan_strategy_management_extra_filters.append(date_filter)
        business_plan_strategy_management_final_filters = " AND ".join(
            business_plan_strategy_management_extra_filters
        )
    else:
        business_plan_strategy_management_final_filters = date_filter

    business_plan_strategy_management_selected_filters = [
        "ID",
        "StrategyID",
        "StatusID",
        "BusinessPlanID",
        "DraftedByID",
        "ProposedByID",
        "ConfidenceID",
        "BeginMonth",
        "Comments",
        "CreationDate",
        "CreationUser",
        "LastModifiedDate",
        "LastModifiedUser",
        "BusinessDate",
        ]

    business_plan_strategy_management = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smebusinessplanstrategymanagement",
        business_plan_strategy_management_final_filters,
        business_plan_strategy_management_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpstrategymanagementstatus ====================================

    business_plan_strategy_management_status_extra_filters = None
    if business_plan_strategy_management_status_extra_filters is not None:
        business_plan_strategy_management_status_extra_filters.append(date_filter)
        business_plan_strategy_management_status_final_filters = " AND ".join(
            business_plan_strategy_management_status_extra_filters
        )
    else:
        business_plan_strategy_management_status_final_filters = date_filter

    business_plan_strategy_management_status_selected_filters = [
        "ID",
        "Label",
        "Priority",
        "BusinessDate",
        ]

    business_plan_strategy_management_status = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpstrategymanagementstatus",
        business_plan_strategy_management_status_final_filters,
        business_plan_strategy_management_status_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpdraftedby ====================================

    business_plan_drafted_by_extra_filters = None
    if business_plan_drafted_by_extra_filters is not None:
        business_plan_drafted_by_extra_filters.append(date_filter)
        business_plan_drafted_by_final_filters = " AND ".join(
            business_plan_drafted_by_extra_filters
        )
    else:
        business_plan_drafted_by_final_filters = date_filter

    business_plan_drafted_by_selected_filters = ["ID", "Label", "BusinessDate"]
    business_plan_drafted_by = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpdraftedby",
        business_plan_drafted_by_final_filters,
        business_plan_drafted_by_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpproposedby ====================================

    business_plan_proposed_by_extra_filters = None
    if business_plan_proposed_by_extra_filters is not None:
        business_plan_proposed_by_extra_filters.append(date_filter)
        business_plan_proposed_by_final_filters = " AND ".join(
            business_plan_proposed_by_extra_filters
        )
    else:
        business_plan_proposed_by_final_filters = date_filter

    business_plan_proposed_by_selected_filters = ["ID", "Label", "BusinessDate"]
    business_plan_proposed_by = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpproposedby",
        business_plan_proposed_by_final_filters,
        business_plan_proposed_by_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpconfidence ====================================

    business_plan_confidence_extra_filters = None
    if business_plan_confidence_extra_filters is not None:
        business_plan_confidence_extra_filters.append(date_filter)
        business_plan_confidence_final_filters = " AND ".join(
            business_plan_confidence_extra_filters
        )
    else:
        business_plan_confidence_final_filters = date_filter

    business_plan_confidence_selected_filters = ["ID", "Label", "BusinessDate"]
    business_plan_confidence = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpconfidence",
        business_plan_confidence_final_filters,
        business_plan_confidence_selected_filters,
        )

    # ============================== cepalplatform_smerfdtbpstrategy ====================================

    business_plan_strategy_extra_filters = None
    if business_plan_strategy_extra_filters is not None:
        business_plan_strategy_extra_filters.append(date_filter)
        business_plan_strategy_final_filters = " AND ".join(
            business_plan_strategy_extra_filters
        )
    else:
        business_plan_strategy_final_filters = date_filter

    business_plan_strategy_selected_filters = ["ID", "Label", "BusinessDate"]
    business_plan_strategy = load_landed_delta_table(
        spark,
        catalog_landed,  
        "cepalplatform_smerfdtbpstrategy",
        business_plan_strategy_final_filters,
        business_plan_strategy_selected_filters,
        )

    # ============================== joins ====================================

    business_plan_strategy_management_joins = (
        business_plan_strategy_management
        .join(
            business_plan_strategy_management_status,
            business_plan_strategy_management_status.ID == business_plan_strategy_management.StatusID,
            "left"
            )
        .join(
            business_plan_drafted_by,
            business_plan_drafted_by.ID == business_plan_strategy_management.DraftedByID,
            "left"
            )
        .join(
            business_plan_strategy,
            business_plan_strategy.ID == business_plan_strategy_management.StrategyID,
            "left"
            )
        .join(
            business_plan_proposed_by,
            business_plan_proposed_by.ID == business_plan_strategy_management.ProposedByID,
            "left"
            )
        .join(
            business_plan_confidence,
            business_plan_confidence.ID == business_plan_strategy_management.ConfidenceID,
            "left"
        )

    )

    target_business_plan_strategy_management = business_plan_strategy_management_joins.select(
        business_plan_strategy_management.ID.cast("Int"),
        business_plan_strategy_management.BusinessPlanID.cast("Int"),
        business_plan_strategy_management_status.Label.cast("String").alias("BPSStatus"),
        business_plan_strategy_management_status.Priority.cast("Int").alias("BPSStatusPriority"),
        business_plan_drafted_by.Label.cast("String").alias("DraftBy"),
        business_plan_strategy.Label.cast("String").alias("Strategy"),
        business_plan_strategy_management.BeginMonth.cast("Date"),
        business_plan_proposed_by.Label.cast("String").alias("ProposedBy"),
        business_plan_confidence.Label.cast("String").alias("Confidence"),
        business_plan_strategy_management.Comments.cast("String"),
        business_plan_strategy_management.CreationDate.cast("Timestamp"),
        business_plan_strategy_management.CreationUser.cast("String"),
        business_plan_strategy_management.LastModifiedDate.cast("Timestamp"),
        business_plan_strategy_management.LastModifiedUser.cast("String"),
        business_plan_strategy_management.StrategyID.cast("Int"),
        business_plan_strategy_management.BusinessDate.cast("Date"), 
    )

    return target_business_plan_strategy_management


In [0]:
def fetchMostRecentStrategy(source_table: SparkDataFrame) -> SparkDataFrame:
    window_spec = Window.partitionBy(
        "BusinessPlanID",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_StrategyType"
    ).orderBy(desc("BP_Strategy_Code"))

    source_table = source_table.withColumn(
        "rnk", row_number().over(window_spec)
    )

    target_table = source_table.filter(col("rnk") == 1)
    return target_table

In [0]:
catalog_landed = "`d-cdh`.landed"
run_date = '2026-02-24' 
indiv_bankruptcy_stand_still = businessPlanIndivBankruptcyStandStill(spark = spark, catalog_landed = catalog_landed, run_date = run_date)
business_plan_corp_bankruptcy = businessPlanCorpBankruptcy(spark = spark, catalog_landed = catalog_landed, run_date = run_date)
tool_forc_cbs = businessPlanToolForc(spark = spark, catalog_landed = catalog_landed, run_date = run_date)
tool_vol_cbs = businessPlanToolVol(spark = spark, catalog_landed = catalog_landed, run_date = run_date)

recent_indiv_bankruptcy_stand_still = fetchMostRecentStrategy(source_table = indiv_bankruptcy_stand_still)
recent_business_plan_corp_bankruptcy = fetchMostRecentStrategy(source_table = business_plan_corp_bankruptcy)
recent_tool_forc_cbs = fetchMostRecentStrategy(source_table = tool_forc_cbs)
recent_tool_vol_cbs = fetchMostRecentStrategy(source_table = tool_vol_cbs)

business_plan_rawdata = (
    recent_indiv_bankruptcy_stand_still
    .union(recent_business_plan_corp_bankruptcy)
    .union(recent_tool_forc_cbs)
    .union(recent_tool_vol_cbs)
)


business_plan = businessPlan(spark, catalog_landed, run_date)
business_plan_strategy = businessPlanStrategy(spark, catalog_landed, run_date)

##### 6. Final **primary.Applications** table

In [0]:
catalog_landed = "`d-cdh`.landed"
run_date = '2026-02-24' 
curated_zone = "`p-cdh`.primary"
cluster_columns = ["BusinessDate"]
date_filter = f"businessdate = {run_date!r}"

# ================== business_plan_rawdata ==================
            
business_plan_rawdata_selected_fields = [
            "BusinessPlanID",
            "TimeToAuction",
            "TimeToRecovery",
            "TimeToSell",
            "PrepaymentMonth",
            "AuctionPrice",
            "BalloonAmount",
            "PrepaymentAmount",
            "Interest",
            "InterestPaymentPeriodicityID",
            "NetRecoveryAmount",
            "BP_Strategy_Code",
            "DebtorCDI",
            "AgreementNumber",
            "BusinessPlan_Status",
            "BusinessPlan_SolutionType",
            "BusinessPlanStrategy_Status",
            "BusinessPlanStrategy_StrategyType",
            "BusinessPlanStrategy_SolutionType",
            "NumberOfInstallments",
            "ImplementationStatus", 
            "FirstInstallmentDate", 
            "StartingCapitalAmount", 
            "tbl",
            "PeriodicityID",
        ]

indiv_bankruptcy_stand_still = businessPlanIndivBankruptcyStandStill(spark, catalog_landed, run_date)
business_plan_corp_bankruptcy = businessPlanCorpBankruptcy(spark, catalog_landed, run_date)
tool_forc_cbs = businessPlanToolForc(spark = spark, catalog_landed = catalog_landed, run_date = run_date)
tool_vol_cbs = businessPlanToolVol(spark = spark, catalog_landed = catalog_landed, run_date = run_date)

recent_indiv_bankruptcy_stand_still = fetchMostRecentStrategy(source_table = indiv_bankruptcy_stand_still)
recent_business_plan_corp_bankruptcy_dedup = fetchMostRecentStrategy(source_table = business_plan_corp_bankruptcy)
recent_tool_forc_cbs_dedup = fetchMostRecentStrategy(source_table = tool_forc_cbs)
recent_tool_vol_cbs_dedup = fetchMostRecentStrategy(source_table = tool_vol_cbs)

business_plan_rawdata = (
    recent_indiv_bankruptcy_stand_still
    .union(recent_business_plan_corp_bankruptcy_dedup)
    .union(recent_tool_forc_cbs_dedup)
    .union(recent_tool_vol_cbs_dedup)
)
business_plan_rawdata = business_plan_rawdata.select(business_plan_rawdata_selected_fields)
business_plan_rawdata = (
    business_plan_rawdata.withColumnRenamed("ImplementationStatus", "implementation status")
                         .withColumnRenamed("FirstInstallmentDate", "first installment date")
)

# ================== business_plan_contract ==================

business_plan_contract_selected_fields = ["Agreement_Number", "BusinessPlanStrategy_ID"]
business_plan_contract = businessPlanContract(spark, catalog_landed, run_date)
business_plan_contract = business_plan_contract.select(business_plan_contract_selected_fields)
business_plan_contract_distinct = business_plan_contract.distinct()

# ================== business_plan_strategy ==================

business_plan_strategy_selected_fields = [
            "Comments",
            "StrategyID", 
            "BPSStatus", 
            "Strategy",  
            "LastModifiedDate",
            "ProposedBy",
            "Confidence",
            "BusinessPlanID",
            "BeginMonth",
            "CreationUser",
            "ID",
            "BusinessDate",
        ]

business_plan_strategy = businessPlanStrategy(spark, catalog_landed, run_date)
business_plan_strategy = (
    business_plan_strategy.withColumnRenamed("ID", "id_strategy")
                          .withColumnRenamed("BusinessPlanID", "BusinessPlanID_strategy")
                          .withColumnRenamed("StrategyID", "Strategy_id")
                          .withColumnRenamed("BPSStatus", "bps_status")
)

# ================== business_plan ==================

business_plan_selected_fields = [
            "ID",
            "ApplicationDate",
            "IntentionDate",
            "DebtorCDI",
            "DebtorAFM",
            "GroupCode",
            "SolutionType", 
            "Business_plan_status", 
            "Comments", 
            "CreationDate", 
            "CreationUser", 
            "LastModifiedDate", 
            "LastModifiedUser", 
            "CompletionTargetDate",
        ]


business_plan = businessPlan(spark, catalog_landed, run_date)
business_plan = (
            business_plan.withColumnRenamed("LastModifiedDate", "LastmodifiedDate_bp")
                         .withColumnRenamed("CreationUser", "creationuser_bp")
                         .withColumnRenamed("Comments", "comments_bp")
                         .withColumnRenamed("CreationDate", "creationdate_bp")
                         .withColumn("DebtorCDI_prefix", substring("DebtorCDI", 1, 3))
        )

# ================== business_plan_lbspaymentschedule ==================

business_plan_lbspaymentschedule_selected_fields = [
            "InstallmentSN",
            "InstallmentTotalAmount",
            "InstallmentDate",
            "InstallmentCapitalAmount",
            "InstallmentChargeAmount",
            "InstallmentN128Amount",
            "InstallmentTotalChargeAmount",
            "InstallmentRemainingCapitalAmount",
            "InstallmentNPVAmount",
            "InstallmentNPVCalculationDate",
            "IsActive",
            "LBSParentID",
            "creationdate", 
        ]

business_plan_lbspaymentschedule = lbsPaymentsSchedule(spark, catalog_landed, run_date)
business_plan_lbspaymentschedule = business_plan_lbspaymentschedule.select(business_plan_lbspaymentschedule_selected_fields)

# ================== portfolio ==================

portfolio_extra_filters = None

if portfolio_extra_filters is not None:
    portfolio_extra_filters.append(date_filter)
    portfolio_final_filters = " AND ".join(portfolio_extra_filters)
else:
    portfolio_final_filters = date_filter

    portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

    portfolio = load_landed_portfolio_table( 
            spark,
            curated_zone, 
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

# ================== joins ==================

subquery_lbs = business_plan_lbspaymentschedule.alias("A").join(
            business_plan_lbspaymentschedule.groupBy("LBSParentID", "InstallmentSN")
            .agg(max("creationdate").alias("creationdate"))
            .alias("B"),
            (col("A.LBSParentID") == col("B.LBSParentID"))
            & (col("A.InstallmentSN") == col("B.InstallmentSN"))
            & (col("A.creationdate") == col("B.creationdate")),
        )

application_sme_df = (
            business_plan_strategy.join(
                business_plan,
                business_plan_strategy["BusinessPlanID_strategy"] == business_plan["ID"],
                "inner",
            )
            .join(
                portfolio,
                portfolio["Prefix"] == business_plan["DebtorCDI_prefix"],
                "left",
            )
            .join(
                business_plan_contract_distinct,
                business_plan_strategy["id_strategy"] == business_plan_contract_distinct["BusinessPlanStrategy_ID"],
                "left",
            )
            .join(
                business_plan_rawdata,
                (
                    coalesce(business_plan_contract_distinct["Agreement_Number"], lit("1"))
                    == coalesce(business_plan_rawdata["AgreementNumber"], lit("1"))
                )
                & (
                    coalesce(business_plan_strategy["Strategy"], lit("1"))
                    == coalesce(business_plan_rawdata["BusinessPlanStrategy_StrategyType"], lit("1"),)
                )
                & (
                    coalesce(business_plan_strategy["bps_status"], lit("1"))
                    == coalesce(business_plan_rawdata["BusinessPlanStrategy_Status"], lit("1"))
                )
                & (business_plan["DebtorCDI"] == business_plan_rawdata["DebtorCDI"])
                & (business_plan_strategy["BusinessPlanID_strategy"] == business_plan_rawdata["BusinessPlanId"]),
                "left",
            )
            .join(
                subquery_lbs,
                business_plan_strategy["BusinessPlanID_strategy"] == subquery_lbs["A.LBSParentID"],
                "left",
            )
        )

application_sme_df = (
            application_sme_df.withColumn(
                "AgreementId",
                coalesce(application_sme_df["Agreement_Number"], lit("-1")),
            )
            .withColumn(
                "ApplicationCancelledDate",
                when(
                    application_sme_df["bps_status"] == "Cancelled",
                    application_sme_df["LastModifiedDate"],
                ).otherwise(lit(None)),
            )
            .withColumn("creationdate_b", application_sme_df["B.creationdate"])
            .withColumn("InstallmentSN_b", application_sme_df["B.InstallmentSN"])
            .withColumn("LBSParentID_b", application_sme_df["B.LBSParentID"])
            .withColumn(
                "IsPrimaryRowFlag",
                when(col("B.InstallmentSN").isNull(), 1).otherwise(0),
            )
        )

# ================== application_sme_df ==================

application_sme_primary_key_fields = [
            business_plan_strategy.BusinessPlanID_strategy,
            application_sme_df.InstallmentSN_b,
            application_sme_df.id_strategy,
            lit("sme"),
        ]

applicationkey_sme_primary_key_fields = [
            business_plan_strategy.BusinessPlanID_strategy,
            application_sme_df.InstallmentSN_b,
            application_sme_df.id_strategy,
        ]

application_sme_df = application_sme_df.withColumn(
            "ApplicationIdKey",
            sha2(concat_ws(";", *application_sme_primary_key_fields), 256),
        ).withColumn(
            "ApplicationKey",
            concat_ws(";", *applicationkey_sme_primary_key_fields),
        )

application_sme_df = application_sme_df.select(
            application_sme_df.ApplicationIdKey.alias("ApplicationId"),
            lit("sme").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchID"),
            application_sme_df.ApplicationKey.cast("string").alias("ApplicationKey"),
            application_sme_df.id_strategy.cast("string").alias("ApplicationStrategyKey"),
            application_sme_df.LBSParentID_b.cast("string").alias("ApplicationScheduleKey"),
            lit(None).alias("ExposureKey"),
            application_sme_df.AgreementId.cast("string").alias("AgreementKey"),
            business_plan.DebtorCDI.alias("CDI"),
            lit(None).alias("PartyKey"),
            application_sme_df.DebtorAFM.alias("TaxRegistrationNumber"),
            application_sme_df.GroupCode.alias("GroupCode"),
            application_sme_df.ID.alias("BusinessPlanKey"),
            application_sme_df.SolutionType.alias("BusinessPlanSolutionTypeCode"),
            application_sme_df.SolutionType.alias("BusinessPlanSolutionType"),
            application_sme_df.Business_plan_status.alias("BusinessPlanSolutionStatusCode"),
            application_sme_df.Business_plan_status.alias("BusinessPlanSolutionStatus"),
            application_sme_df.comments_bp.alias("BusinessPlanComments"),
            application_sme_df.creationdate_bp.alias("BusinessPlanCreatedDate"),
            application_sme_df.creationuser_bp.alias("BusinessPlanCreatedBy"),
            application_sme_df.LastmodifiedDate_bp.alias("BusinessPlanLastUpdatedDate"),
            application_sme_df.LastModifiedUser.alias("BusinessPlanLastUpdatedBy"),
            application_sme_df.Strategy_id.cast("string").alias("ApplicationTypeCode"),
            application_sme_df.Strategy.cast("string").alias("ApplicationType"),
            application_sme_df.bps_status.cast("string").alias("ApplicationStatusCode"),
            application_sme_df.bps_status.cast("string").alias("ApplicationStatus"),
            application_sme_df.ApplicationDate.cast("timestamp").alias("ApplicationDate"),
            application_sme_df.IntentionDate.cast("date").alias("IntentionDate"),
            lit(None).alias("ApprovedDate"),
            lit(None).alias("RejectedDate"),
            application_sme_df.ApplicationCancelledDate.cast("timestamp").alias("CancelledDate"),
            application_sme_df.CompletionTargetDate.cast("string").alias("ImplementationDate"),
            application_sme_df.BeginMonth.alias("BeginDate"),
            application_sme_df.ProposedBy.alias("ProposedTypeCode"),
            application_sme_df.ProposedBy.alias("ProposedType"),
            application_sme_df.Confidence.alias("ConfidenceTypeCode"),
            application_sme_df.Confidence.alias("ConfidenceType"),
            lit(None).alias("Installment"),
            lit(None).alias("PropositionUser"),
            lit(None).alias("Tenor"),
            application_sme_df.TimeToAuction.alias("TimeToAuction"),
            application_sme_df.TimeToRecovery.alias("TimeToRecovery"),
            application_sme_df.TimeToSell.alias("TimeToSell"),
            application_sme_df.PrepaymentMonth.alias("PrepaymentDate"),
            application_sme_df.AuctionPrice.alias("AuctionPrice"),
            application_sme_df.BalloonAmount.alias("BalloonAmount"),
            application_sme_df.PrepaymentAmount.alias("PrepaymentAmount"),
            application_sme_df.BusinessPlanStrategy_SolutionType.alias("BusinessPlanStrategySolutionTypeCode"),
            application_sme_df.BusinessPlanStrategy_SolutionType.alias("BusinessPlanStrategySolutionType"),
            application_sme_df.Interest.alias("Interest"),
            application_sme_df.InterestPaymentPeriodicityID.alias("InterestPaymentPeriodicity"),
            application_sme_df.NetRecoveryAmount.alias("NetRecoveryAmount"),
            application_sme_df.BP_Strategy_Code.alias("BusinessPlanStrategyCode"),
            application_sme_df.Comments.alias("Comments"),
            application_sme_df.LastModifiedDate.alias("LastUpdatedDate"),
            application_sme_df.creationdate_b.alias("CreatedDate"),
            application_sme_df.CreationUser.cast("string").alias("CreatedBy"),
            lit(None).alias("AssigneeUserCode"),
            lit(None).alias("AssigneeUser"),
            lit(None).alias("AssigneePortfolioGroupCode"),
            lit(None).alias("AssigneePortfolioGroup"),
            lit(None).alias("IntegrationCode"),
            lit(None).alias("CommunicationChannelCode"),
            lit(None).alias("CommunicationChannel"),
            lit(None).alias("FlowSegmentCode"),
            lit(None).alias("FlowSegment"),
            lit(None).alias("UserCreatedQualcoGroupCode"),
            lit(None).alias("ContractSigningMethodCode"),
            lit(None).alias("ContractSigningMethod"),
            lit(None).alias("ManagedByPortfolioGroupCode"),
            lit(None).alias("InterbankFlag"),
            lit(None).alias("OcwFlag"),
            lit(None).alias("ContactSigningPlaceTypeCode"),
            lit(None).alias("ContactSigningPlaceType"),
            lit(None).alias("ReportingApplicationStatusCode"),
            lit(None).alias("ReportingApplicationStatus"),
            lit(None).alias("ClientResponseCode"),
            lit(None).alias("ClientResponse"),
            lit(None).alias("OcwSimulationVote"),
            lit(None).alias("StartedDate"),
            lit(None).alias("SubmittedDate"),
            lit(None).alias("QualcoAppPreApprovedDate"),
            lit(None).alias("QualcoAppDisbursedDate"),
            lit(None).alias("QualcoAppFinalStepDate"),
            lit(None).alias("UserCreatedChannelCode"),
            lit(None).alias("UserUnit"),
            lit(None).alias("UserCreatedGroupCode"),
            lit(None).alias("UserCreatedGroup"),
            lit(None).alias("ApplicationImplementationStatusCode"),
            lit(None).alias("ApplicationImplementationStatus"),
            lit(None).alias("ApplicationActivationDate"),
            lit(None).alias("ContactSigningPlaceCode"),
            lit(None).alias("PolicyOverrideFlag"),
            application_sme_df.InstallmentSN_b.alias("InstallmentNumber"),
            application_sme_df.InstallmentTotalAmount.alias("InstallmentAmount"),
            application_sme_df.InstallmentDate.alias("InstallmentDate"),
            application_sme_df.InstallmentCapitalAmount.alias("InstallmentCapitalAmount"),
            application_sme_df.InstallmentChargeAmount.alias("InstallmentChargeAmount"),
            application_sme_df.InstallmentN128Amount.alias("InstallmentN128Amount"),
            application_sme_df.InstallmentTotalChargeAmount.alias("InstallmentTotalChargeAmount"),
            application_sme_df.InstallmentRemainingCapitalAmount.alias("InstallmentRemainingCapitalAmount"),
            application_sme_df.InstallmentNPVAmount.alias("InstallmentNPVAmount"),
            application_sme_df.InstallmentNPVCalculationDate.alias("InstallmentNPVCalculationDate"),
            application_sme_df.IsActive.alias("InstallementStatusFlag"),
            application_sme_df.IsPrimaryRowFlag,
            application_sme_df.BusinessDate,
        )

##### Checks

%md
##### cepalplatform_smevbusinessplandatacorpbankruptsy SmeVBusinessPlanDataCorpBankruptsy | Okay
##### cepalplatform_smevbusinessplandataforccbsnonre SmeVBusinessPlanDataForcCbsNonRe | Okay
##### cepalplatform_smevbusinessplandataforccbsre3rdparty SmeVBusinessPlanDataForcCbsRe3rdParty | Okay
##### cepalplatform_smevbusinessplandataforccbsreohold SmeVBusinessPlanDataForcCbsReoHold | Okay
##### cepalplatform_smevbusinessplandataforccbsreosale SmeVBusinessPlanDataForcCbsReoSale | Okay

##### cepalplatform_smevbusinessplandatavolcbsnonre SmeVBusinessPlanDataVolCbsNonRe | Okay
##### cepalplatform_smevbusinessplandatavolcbsre3rdparty SmeVBusinessPlanDataVolCbsRe3rdParty | Okay
##### cepalplatform_smevbusinessplandatavolcbsreohold SmeVBusinessPlanDataVolCbsReoHold | Okay
##### cepalplatform_smevbusinessplandatavolcbsreosale SmeVBusinessPlanDataVolCbsReoSale | Okay

##### cepalplatform_smevbusinessplandataindivbankruptcy SmeVBusinessPlanDataIndivBankruptcy | Okay
##### cepalplatform_smevbusinessplandatastandstill SmeVBusinessPlanDataStandStill | Okay

##### cepalplatform_smevbusinessplandatacorpbankruptsy SmeVBusinessPlanDataCorpBankruptsy | Okay
##### cepalplatform_smevbusinessplandataforccbsnonre SmeVBusinessPlanDataForcCbsNonRe | Okay
##### cepalplatform_smevbusinessplandataforccbsre3rdparty SmeVBusinessPlanDataForcCbsRe3rdParty | Okay
##### cepalplatform_smevbusinessplandataforccbsreohold SmeVBusinessPlanDataForcCbsReoHold | Okay
##### cepalplatform_smevbusinessplandataforccbsreosale SmeVBusinessPlanDataForcCbsReoSale | Okay

##### cepalplatform_smevbusinessplandatavolcbsnonre SmeVBusinessPlanDataVolCbsNonRe | Okay
##### cepalplatform_smevbusinessplandatavolcbsre3rdparty SmeVBusinessPlanDataVolCbsRe3rdParty | Okay
##### cepalplatform_smevbusinessplandatavolcbsreohold SmeVBusinessPlanDataVolCbsReoHold | Okay
##### cepalplatform_smevbusinessplandatavolcbsreosale SmeVBusinessPlanDataVolCbsReoSale | Okay

##### cepalplatform_smevbusinessplandataindivbankruptcy SmeVBusinessPlanDataIndivBankruptcy | Okay
##### cepalplatform_smevbusinessplandatastandstill SmeVBusinessPlanDataStandStill | Okay

##### SmeBpsmLbsPaymentSchedule cepalplatform_smebpsmlbspaymentschedule | Okay

##### SmeBusinessPlanContract cepalplatform_smebusinessplancontract | Okay
##### SmeBusinessPlanContractLoan cepalplatform_smebusinessplancontractloan | Okay
##### SmeBusinessPlanContractCollateral cepalplatform_smebusinessplancontractcollateral | Okay

##### SmeBusinessPlan cepalplatform_smebusinessplan | Okay
##### SmeRfdtBpStatus cepalplatform_smerfdtbpstatus | Okay
##### SmeRfdtBpSolutionType cepalplatform_smerfdtbpsolutiontype | Okay

##### SmeBusinessPlanStrategyManagement cepalplatform_smebusinessplanstrategymanagement | Okay
##### SmeRfdtBpStrategyManagementStatus cepalplatform_smerfdtbpstrategymanagementstatus | Okay
##### SmeRfdtBpDraftedBy cepalplatform_smerfdtbpdraftedby | Okay
##### SmeRfdtBpProposedBy cepalplatform_smerfdtbpproposedby | Okay
##### SmeRfdtBpConfidence cepalplatform_smerfdtbpconfidence | Okay
##### SmeRfdtBpStrategy cepalplatform_smerfdtbpstrategy | Okay

##### **Hypothesis validation:** Entries with higher **BusinessPlanStrategy_Code** have most recent **Modified Business Strategy Type**

In [0]:
%sql
SELECT BusinessPlanStrategy_Code, 
       BusinessPlanStrategy_LastModifiedDate,
       BusinessPlan_Status,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk,
       *
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandataforccbsnonre
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 667
AND AgreementNumber = "NEP1824189091-0000000000349993"
AND BusinessPlan_Status = "Intention"
AND BusinessPlanStrategy_StrategyType = "Forced CBS Non-RE"
ORDER BY BusinessPlanStrategy_LastModifiedDate DESC

In [0]:
%sql
SELECT BusinessPlanStrategy_Code,
       BusinessPlanStrategy_LastModifiedDate,
       BusinessPlan_Status,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk,
       *
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandataforccbsre3rdparty
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 136
AND AgreementNumber = 'SYM1537001169'
AND BusinessPlan_Status = 'Cancelled'
AND BusinessPlanStrategy_StrategyType = 'Forced CBS RE 3rd Party'
ORDER BY BusinessPlanStrategy_LastModifiedDate DESC


In [0]:
%sql
SELECT BusinessPlanStrategy_Code,
       BusinessPlanStrategy_LastModifiedDate,
       BusinessPlanStrategy_Code,
       BusinessPlan_Status,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk,
       *
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandataforccbsreosale
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 1091
AND AgreementNumber = 'JUP1831516591-0000000000397362'
AND BusinessPlan_Status = 'Cancelled'
AND BusinessPlanStrategy_StrategyType = 'Forced CBS REO Sale'
ORDER BY BusinessPlanStrategy_LastModifiedDate DESC

In [0]:
%sql
SELECT BusinessPlanStrategy_Code,
       LastModifiedDate,
       BusinessPlanStrategy_Code,
       BusinessPlan_Status,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk,
       *
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandatastandstill
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 2075
AND AgreementNumber = 'NEP1821142491-0000000000514178'
AND BusinessPlan_Status = 'Intention'
AND BusinessPlanStrategy_StrategyType = 'Stand Still'
ORDER BY LastModifiedDate DESC

In [0]:
%sql
SELECT BusinessPlanStrategy_Code, 
       BusinessPlanStrategy_LastModifiedDate,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID,
                                        AgreementNumber,
                                        BusinessPlan_Status,
                                        BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk, *
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandatavolcbsreosale
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 1067
AND AgreementNumber = "JUP1806463991-0000000001260116"
AND BusinessPlan_Status = "Intention"
AND BusinessPlanStrategy_StrategyType = "Voluntary CBS REO Sale"
ORDER BY BusinessPlanStrategy_LastModifiedDate DESC

##### 1. Checks for the **BusinessPlanStrategy_LastModifiedDate** column

In [0]:
%sql
WITH source_table AS (
SELECT BusinessPlanStrategy_LastModifiedDate,
       BusinessPlan_ID,
       AgreementNumber,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType,
       BusinessPlanStrategy_Code,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID,
                                        AgreementNumber,
                                        BusinessPlan_Status,
                                        BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandataforccbsre3rdparty 
WHERE BusinessDate = '2026-02-24'
),

with_most_recent_strategy AS (
SELECT *
FROM source_table
WHERE rnk = 1
),

with_older_strategies AS (
SELECT BusinessPlan_ID,
         AgreementNumber,
         BusinessPlan_Status,
         BusinessPlanStrategy_StrategyType,
         MIN(BusinessPlanStrategy_LastModifiedDate) AS OlderBusinessPlanStrategy_LastModifiedDate
FROM source_table
WHERE rnk <> 1
GROUP BY BusinessPlan_ID,
         AgreementNumber,
         BusinessPlan_Status,
         BusinessPlanStrategy_StrategyType
),

-- QUESTION: Υπάρχει Startegy της στήλης: OlderBusinessPlanStrategy_LastModifiedDate που είναι νεότερο από το strategy της στήλης: BusinessPlanStrategy_LastModifiedDate?
results AS (
                SELECT *
                FROM (
                        SELECT COALESCE(y.OlderBusinessPlanStrategy_LastModifiedDate,
                                        x.BusinessPlanStrategy_LastModifiedDate) AS OlderBusinessPlanStrategy_LastModifiedDate, x.*
                        FROM with_most_recent_strategy AS x
                        LEFT JOIN with_older_strategies AS y
                        ON x.BusinessPlan_ID = y.BusinessPlan_ID
                        AND x.AgreementNumber  = y.AgreementNumber
                        AND x.BusinessPlan_Status = y.BusinessPlan_Status
                        AND x.BusinessPlanStrategy_StrategyType = y.BusinessPlanStrategy_StrategyType
                        )
                WHERE OlderBusinessPlanStrategy_LastModifiedDate > BusinessPlanStrategy_LastModifiedDate
                )

-- Specific counter-example
SELECT BusinessPlanStrategy_LastModifiedDate, BusinessPlanStrategy_Code, *
FROM source_table
WHERE BusinessPlan_ID = 165
AND AgreementNumber  = "EUC0910010634"
AND BusinessPlan_Status = "Intention"
AND BusinessPlanStrategy_StrategyType = "Forced CBS RE 3rd Party"
ORDER BY BusinessPlanStrategy_LastModifiedDate DESC

In [0]:
%sql
-- BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
SELECT BusinessPlan_ID,
       AgreementNumber,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType, 
       COUNT(CollectionPlan_Month) AS CountMonths
FROM (
    SELECT *,
          ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID,
                                            AgreementNumber,
                                            BusinessPlan_Status,
                                            BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
    FROM `d-cdh`.landed.cepalplatform_smevbusinessplandatacorpbankruptsy
    WHERE BusinessDate = '2026-02-24'
    )
WHERE rnk = 1
GROUP BY BusinessPlan_ID,
         AgreementNumber,
         BusinessPlan_Status,
         BusinessPlanStrategy_StrategyType
ORDER BY CountMonths DESC
LIMIT 10

In [0]:
%sql
SELECT *, ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID,
                                            AgreementNumber,
                                            BusinessPlan_Status,
                                            BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandatacorpbankruptsy
WHERE BusinessDate = '2026-02-24'
AND BusinessPlan_ID = 172
AND AgreementNumber = "EUC9707007981"
AND BusinessPlan_Status = "Intention"
AND BusinessPlanStrategy_StrategyType = "Corp. Bankruptcy (art.99, art.106)"
ORDER BY CollectionPlan_Month DESC
LIMIT 10


In [0]:
%sql
WITH business_plan_corp_bankruptsy AS ( 
SELECT *, 
       ROW_NUMBER () OVER (PARTITION BY BusinessPlan_ID,
                                        AgreementNumber,
                                        BusinessPlan_Status,
                                        BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
FROM `d-cdh`.landed.cepalplatform_smevbusinessplandatacorpbankruptsy
WHERE BusinessDate = '2026-02-24'
)

SELECT CollectionPlan_Month, CollectionPlan_Amount, *
FROM business_plan_corp_bankruptsy
WHERE AgreementNumber = 'EUC9707007981'
AND BusinessPlan_ID = 172
AND BusinessPlanStrategy_StrategyType = 'Corp. Bankruptcy (art.99, art.106)'
AND BusinessPlan_Status = 'Intention'
ORDER BY CollectionPlan_Month DESC

In [0]:
%sql
SELECT *
FROM `p-cdh`.landed.sme_business_plan_strategy
WHERE BusinessDate = '2026-02-24'
AND BusinessPlanID = 172
